In [1]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import mannwhitneyu
from tqdm import trange
import matplotlib.ticker as ticker
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

#DATA_PATH1 = "../../0-download-inputs/data-files/"
DATA_PATH2 = "../../18-finalize/processed-data/"
DATA_PATH3 = "../../17-meltome-atlas/processed-data/"
MS_COLOR   = "#DDAA33"
Y2H_COLOR  = "#BB5566"

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 14,
    "axes.titlepad": 25,
    "font.family": "sans-serif",
    "font.sans-serif": "Arial",
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

## AE-MS

In [2]:
nodes        = pd.read_pickle(f"{DATA_PATH2}20260210-s288c-ANnnotated-Yeast-Interactome.pkl")

# determine which nodes are hubs based on percentile cutoff for degree centrality
cut          = 0.10
ms_threshold = nodes['degree_centrality'].quantile(1 - cut)

print(f"Threshold for defining a hub in MS data is {ms_threshold:.5f}")

summary_rows = []

def record_stage(df, stage, previous_count):
    summary_rows.append({
        "stage": stage,
        "total_remaining": len(df),
        "removed_at_step": previous_count - len(df),
        "hubs": int((df["degree_centrality"] >= ms_threshold).sum()),
        "non_hubs": int((df["degree_centrality"] < ms_threshold).sum()),
        "missing_degree": int(df["degree_centrality"].isna().sum()),
        "ghosh_dill_nonnull": int(df["Ghosh-Dill-dG"].notna().sum()),
        "cagiada_nonnull": int(df["cagiada-dG"].notna().sum()),
        "rosetta_nonnull": int(df["Rosetta_best_total_score"].notna().sum()),
    })

    # Retain class counts for optional inclusion in the figure.
    for category, count in (
        df["DeepTMHMM_class"].fillna("Missing").value_counts().items()
    ):
        summary_rows[-1][f"class_{category}"] = int(count)


filtered_nodes = nodes.copy()
record_stage(filtered_nodes, "Before filtering", len(filtered_nodes))

previous_count = len(filtered_nodes)
filtered_nodes = filtered_nodes[
    filtered_nodes["mean_plddt"] >= 70
].copy()
record_stage(filtered_nodes, "After pLDDT >= 70", previous_count)

for category in ["TM", "SP+TM", "BETA"]:
    previous_count = len(filtered_nodes)
    filtered_nodes = filtered_nodes[
        ~filtered_nodes["DeepTMHMM_class"].isin([category])
    ].copy()
    record_stage(
        filtered_nodes,
        f"After removing {category}",
        previous_count,
    )

filtering_summary = pd.DataFrame(summary_rows).fillna(0)
count_columns = filtering_summary.columns.drop("stage")
filtering_summary[count_columns] = filtering_summary[count_columns].astype(int)

filtering_summary.to_csv("filtering_summary.csv", index=False)
display(filtering_summary)

Threshold for defining a hub in MS data is 0.01182


,stage,total_remaining,removed_at_step,hubs,non_hubs,missing_degree,ghosh_dill_nonnull,cagiada_nonnull,rosetta_nonnull,class_GLOB,class_TM,class_SP,class_SP+TM,class_Missing,class_BETA
0,Before filtering,3927,0,393,3534,0,3357,3357,3357,3273,457,84,61,51,1
1,After pLDDT >= 70,2964,963,365,2599,0,2528,2528,2528,2465,383,63,52,0,1
2,After removing TM,2581,383,360,2221,0,2528,2528,2528,2465,0,63,52,0,1
3,After removing SP+TM,2529,52,357,2172,0,2528,2528,2528,2465,0,63,0,0,1
4,After removing BETA,2528,1,357,2171,0,2528,2528,2528,2465,0,63,0,0,0


## Y2H

In [3]:
nodes = pd.read_pickle(f"{DATA_PATH3}y2h-test-s288c-step17.pkl")

# determine which nodes are hubs based on percentile cutoff for degree centrality
cut          = 0.10
ms_threshold = nodes['degree_centrality'].quantile(1 - cut)

print(f"Threshold for defining a hub in MS data is {ms_threshold:.5f}")

summary_rows = []

def record_stage(df, stage, previous_count):
    summary_rows.append({
        "stage": stage,
        "total_remaining": len(df),
        "removed_at_step": previous_count - len(df),
        "hubs": int((df["degree_centrality"] >= ms_threshold).sum()),
        "non_hubs": int((df["degree_centrality"] < ms_threshold).sum()),
        "missing_degree": int(df["degree_centrality"].isna().sum()),
        "ghosh_dill_nonnull": int(df["Ghosh-Dill-dG"].notna().sum()),
        "cagiada_nonnull": int(df["cagiada-dG"].notna().sum()),
        "rosetta_nonnull": int(df["Rosetta_best_total_score"].notna().sum()),
    })

    # Retain class counts for optional inclusion in the figure.
    for category, count in (
        df["DeepTMHMM_class"].fillna("Missing").value_counts().items()
    ):
        summary_rows[-1][f"class_{category}"] = int(count)


filtered_nodes = nodes.copy()
record_stage(filtered_nodes, "Before filtering", len(filtered_nodes))

previous_count = len(filtered_nodes)
filtered_nodes = filtered_nodes[
    filtered_nodes["mean_plddt"] >= 70
].copy()
record_stage(filtered_nodes, "After pLDDT >= 70", previous_count)

for category in ["TM", "SP+TM", "BETA"]:
    previous_count = len(filtered_nodes)
    filtered_nodes = filtered_nodes[
        ~filtered_nodes["DeepTMHMM_class"].isin([category])
    ].copy()
    record_stage(
        filtered_nodes,
        f"After removing {category}",
        previous_count,
    )

filtering_summary = pd.DataFrame(summary_rows).fillna(0)
count_columns = filtering_summary.columns.drop("stage")
filtering_summary[count_columns] = filtering_summary[count_columns].astype(int)

filtering_summary.to_csv("filtering_summary.csv", index=False)
display(filtering_summary)

Threshold for defining a hub in MS data is 0.00297


,stage,total_remaining,removed_at_step,hubs,non_hubs,missing_degree,ghosh_dill_nonnull,cagiada_nonnull,rosetta_nonnull,class_GLOB,class_TM,class_Missing,class_SP,class_SP+TM
0,Before filtering,2018,0,223,1795,0,1752,1752,1752,1724,193,58,33,10
1,After pLDDT >= 70,1426,592,167,1259,0,1268,1268,1268,1245,151,0,23,7
2,After removing TM,1275,151,150,1125,0,1268,1268,1268,1245,0,0,23,7
3,After removing SP+TM,1268,7,150,1118,0,1268,1268,1268,1245,0,0,23,0
4,After removing BETA,1268,0,150,1118,0,1268,1268,1268,1245,0,0,23,0
